# RISKTWIN · Bloque URB-02 — Prueba piloto
## Grafo urbano para exposición, conectividad y criticidad en zonas de riesgo (Guayaquil)

**Universidad de Guayaquil · Facultad de Ingeniería Industrial**
**Autores:** Macias Carranza Krystel Shanely · Suarez Barco José Enrique

---

### ¿Qué hace este notebook?
Es una **prueba piloto** de punta a punta del flujo definido en el anteproyecto (DSR + KGC Pipeline). Sobre un sector acotado de Guayaquil:

| Fase | Contenido | Responsable |
|---|---|---|
| 1 | Adquisición de la red vial desde OpenStreetMap (OSMnx) | Krystel |
| 2 | Preprocesamiento y limpieza topológica | Krystel |
| 3 | Construcción del grafo y métricas de conectividad y centralidad | Krystel |
| 4 | Variables de exposición (puntos de interés y densidad edificada) | José |
| 5 | Índice de criticidad, correlación de Spearman y regresión logística (Alto/Medio/Bajo) | José |
| 6 | Mapas temáticos y exportación de entregables | José |
| 7 | Evaluación y validación (robustez del grafo, reproducibilidad) | Ambos |

### Advertencias metodológicas (leer antes de ejecutar)
1. **Exposición con proxies abiertos.** Sin datos del INEC cargados, la densidad poblacional se aproxima con la densidad de edificaciones de OSM y los puntos de interés (hospitales, escuelas, mercados, etc.). Hay una celda opcional para incorporar datos propios.
2. **Etiquetas Alto/Medio/Bajo.** En el piloto se derivan del propio índice de criticidad (terciles), por lo que el modelo logístico **demuestra el flujo, no valida predicción real**. Para una validación real hay que usar eventos observados (accidentes ATM, delitos ECU911). Existe una celda opcional para ello.
3. Los datos de OSM pueden tener vacíos; la ausencia de un elemento no implica ausencia de riesgo.

## 0. Preparación del entorno
Se instalan las librerías que Colab no trae por defecto (`osmnx`). El resto (`geopandas`, `networkx`, `scikit-learn`, `scipy`, `matplotlib`) ya viene instalado.

In [ ]:
!pip install -q osmnx

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, random
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA)

ox.settings.use_cache = True
ox.settings.log_console = False
print("OSMnx", ox.__version__, "| NetworkX", nx.__version__, "| GeoPandas", gpd.__version__)

### Parámetros del piloto
Aquí se define **todo lo configurable**. Para cambiar de zona basta con modificar el centro y el radio.

- `CENTRO`: coordenadas (latitud, longitud) del centro del sector. Por defecto, un punto en el **Guasmo** (sur de Guayaquil). *Ajústalo con las coordenadas del sector que el equipo decida; puedes copiarlas desde Google Maps (clic derecho → coordenadas).*
- `RADIO_M`: radio de análisis en metros. 1500 m es liviano para Colab; súbelo a 2500–3000 m cuando todo funcione.
- `RADIO_EXPOSICION_M`: distancia a la que se cuentan POIs y edificios alrededor de cada nodo.
- Pesos del índice: se documentan para que sean fáciles de justificar y modificar (análisis de sensibilidad).

In [ ]:
# --- Zona de estudio ---
NOMBRE_ZONA = "Guasmo (piloto)"
CENTRO = (-2.2450, -79.8900)   # (lat, lon) -> AJUSTAR según el sector elegido
RADIO_M = 1500

# --- Exposición ---
RADIO_EXPOSICION_M = 300

# --- Pesos del índice de criticidad ---
# Componente estructural (de la red) y componente de exposición (del contexto)
PESO_ESTRUCTURAL = 0.5
PESO_EXPOSICION = 0.5
# Dentro del componente estructural
W_BET, W_CLO, W_DEG = 0.50, 0.25, 0.25
# Dentro del componente de exposición
W_POI, W_EDI = 0.50, 0.50

# Peso de cada tipo de punto de interés (a más sensible, más peso)
PESO_POI = {
    "hospital": 3.0, "clinic": 2.0, "school": 2.0, "college": 2.0, "university": 2.0,
    "police": 1.5, "fire_station": 1.5, "marketplace": 1.5, "bus_station": 1.0,
}

SALIDA = "salidas_urb02"
os.makedirs(SALIDA, exist_ok=True)
assert abs(PESO_ESTRUCTURAL + PESO_EXPOSICION - 1) < 1e-9
assert abs(W_BET + W_CLO + W_DEG - 1) < 1e-9 and abs(W_POI + W_EDI - 1) < 1e-9

### Conexión opcional con Supabase (PostgreSQL)
Permite **cargar** los accidentes históricos (ATM) desde tu base en Supabase y **guardar** al final los resultados del grafo. Todo es opcional: con `USAR_SUPABASE = False` el notebook corre completo sin credenciales.

**Credenciales (recomendado):** en Colab abre el icono 🔑 *Secretos* y crea `SUPABASE_URL` y `SUPABASE_KEY` (activando "Acceso desde el cuaderno"). Así no quedan escritas en el notebook. *Project Settings → API* en Supabase.

> Usa la clave `anon` para leer y la `service_role` solo si necesitas escribir sin políticas RLS. **Nunca compartas ni subas a GitHub una clave `service_role`.**

In [ ]:
!pip install -q supabase

In [ ]:
import requests
from supabase import create_client, Client

USAR_SUPABASE = False                 # True para activar carga/guardado
TABLA_ACCIDENTES = "nombreDetabla"    # tabla de accidentes en PostgreSQL
COL_LAT, COL_LON = "latitud", "longitud"   # nombres de las columnas de coordenadas
TABLA_SALIDA_NODOS = "urb02_nodos"
TABLA_SALIDA_ARISTAS = "urb02_aristas"
TAM_PAGINA = 1000                     # Supabase devuelve máx. 1000 filas por consulta

def obtener_credenciales():
    """Lee primero los Secretos de Colab, luego variables de entorno."""
    try:
        from google.colab import userdata
        return userdata.get("SUPABASE_URL"), userdata.get("SUPABASE_KEY")
    except Exception:
        return os.environ.get("SUPABASE_URL"), os.environ.get("SUPABASE_KEY")

SUPABASE_URL, SUPABASE_KEY = obtener_credenciales() if USAR_SUPABASE else (None, None)

# ---------- MÉTODO A: API REST con requests (con paginación) ----------
def cargar_datos_api_rest(tabla=TABLA_ACCIDENTES):
    headers = {"apikey": SUPABASE_KEY, "Authorization": f"Bearer {SUPABASE_KEY}"}
    filas, offset = [], 0
    print("📡 Consultando API REST de Supabase...")
    while True:
        url = f"{SUPABASE_URL}/rest/v1/{tabla}?select=*&limit={TAM_PAGINA}&offset={offset}"
        r = requests.get(url, headers=headers, timeout=60)
        if r.status_code != 200:
            print(f"❌ Error en la llamada API: {r.status_code}\n{r.text}")
            return None
        lote = r.json()
        filas += lote
        if len(lote) < TAM_PAGINA:
            break
        offset += TAM_PAGINA
    return pd.DataFrame(filas)

# ---------- MÉTODO B: SDK oficial de Supabase (con paginación) ----------
def cargar_datos_sdk(tabla=TABLA_ACCIDENTES):
    print("📡 Conectando mediante SDK oficial...")
    try:
        cliente: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
        filas, desde = [], 0
        while True:
            resp = cliente.table(tabla).select("*").range(desde, desde + TAM_PAGINA - 1).execute()
            filas += resp.data
            if len(resp.data) < TAM_PAGINA:
                break
            desde += TAM_PAGINA
        return pd.DataFrame(filas)
    except Exception as e:
        print(f"❌ Error usando el SDK: {e}")
        return None

# ---------- GUARDAR resultados en Supabase ----------
def guardar_en_supabase(df, tabla, lote=500):
    """Inserta un DataFrame (sin geometría) por lotes. La tabla debe existir previamente."""
    cliente = create_client(SUPABASE_URL, SUPABASE_KEY)
    limpio = df.copy()
    for c in limpio.columns:
        if limpio[c].dtype == "object" or str(limpio[c].dtype) == "category":
            limpio[c] = limpio[c].astype(str)
    limpio = limpio.replace({np.nan: None})
    registros = limpio.to_dict(orient="records")
    for i in range(0, len(registros), lote):
        cliente.table(tabla).insert(registros[i:i + lote]).execute()
    print(f"✔ {len(registros)} filas guardadas en '{tabla}'")

# ---------- Carga de accidentes ----------
df_accidentes = None
if USAR_SUPABASE:
    df_accidentes = cargar_datos_api_rest()      # o: cargar_datos_sdk()
    if df_accidentes is not None:
        print(f"Accidentes cargados: {len(df_accidentes):,}")
        display(df_accidentes.head())
else:
    print("Supabase desactivado (USAR_SUPABASE = False).")

## Fase 1 · Adquisición y extracción de datos
Se descarga la red **vial para vehículos** (`network_type="drive"`) alrededor del centro definido. OSMnx devuelve un grafo dirigido donde:
- **Nodos** = intersecciones y finales de calle.
- **Aristas** = tramos de calle con atributos (`length`, `highway`, `oneway`, `name`...).

OSMnx ya simplifica la topología por defecto (elimina nodos intermedios que no son intersecciones reales).

In [ ]:
G = ox.graph_from_point(CENTRO, dist=RADIO_M, network_type="drive", simplify=True)
print(f"Zona: {NOMBRE_ZONA}")
print(f"Nodos: {G.number_of_nodes():,} | Aristas: {G.number_of_edges():,}")
fig, ax = ox.plot_graph(G, node_size=6, node_color="#d62728", edge_color="#555555",
                        bgcolor="white", show=False, close=False)
ax.set_title("Fase 1 · Red vial descargada de OpenStreetMap")
plt.show()

## Fase 2 · Preprocesamiento y limpieza
Objetivo: obtener un grafo **coherente y sin sesgos topológicos**. Se aplican tres controles:

1. **Proyección a metros** (UTM): necesaria para medir distancias y áreas correctamente.
2. **Componente fuertemente conectado más grande**: elimina tramos "huérfanos" a los que no se puede llegar o de los que no se puede salir (frecuentes en bordes del recorte).
3. **Eliminación de bucles** (aristas que conectan un nodo consigo mismo), que distorsionan el grado.

In [ ]:
# 1) Proyección a UTM (metros)
Gp = ox.project_graph(G)
CRS_M = Gp.graph["crs"]

# 2) Componente fuertemente conectado más grande
n_antes = Gp.number_of_nodes()
Gp = ox.truncate.largest_component(Gp, strongly=True)

# 3) Bucles
bucles = list(nx.selfloop_edges(Gp))
Gp.remove_edges_from(bucles)

print(f"Nodos antes: {n_antes:,} -> después: {Gp.number_of_nodes():,}")
print(f"Bucles eliminados: {len(bucles)}")
print("¿Fuertemente conexo?:", nx.is_strongly_connected(Gp))

# Resumen de calidad del grafo
est = ox.stats.basic_stats(Gp, circuity_dist="euclidean")
resumen = {k: est[k] for k in ["n", "m", "k_avg", "edge_length_total", "edge_length_avg",
                               "streets_per_node_avg", "intersection_count", "circuity_avg"]}
pd.Series(resumen).round(2).to_frame("valor")

## Fase 3 · Construcción del grafo y métricas estructurales
Se calculan sobre una versión **no dirigida y ponderada por longitud (metros)**:

| Métrica | Interpretación |
|---|---|
| **Grado** | Número de conexiones de la intersección. |
| **Centralidad de intermediación (betweenness)** | Frecuencia con la que un nodo/tramo está en los caminos más cortos: *puentes* de la red. |
| **Centralidad de cercanía (closeness)** | Qué tan accesible es el nodo respecto al resto de la red. |

> En redes con ~1 000–2 500 nodos el cálculo exacto toma segundos. Si amplías el radio mucho, usa `k=` (muestreo) en `betweenness_centrality`.

In [ ]:
# Versión no dirigida, simple, ponderada por longitud (mínima entre paralelas)
Gs = ox.convert.to_digraph(Gp, weight="length").to_undirected()

grado = dict(Gs.degree())
bet = nx.betweenness_centrality(Gs, weight="length", normalized=True)
clo = nx.closeness_centrality(Gs, distance="length")

nodos, aristas = ox.convert.graph_to_gdfs(Gp)
nodos = nodos.copy()
nodos["grado"] = pd.Series(grado)
nodos["betweenness"] = pd.Series(bet)
nodos["closeness"] = pd.Series(clo)

nodos[["grado", "betweenness", "closeness"]].describe().round(4)

In [ ]:
# Betweenness de aristas (tramos)
eb = nx.edge_betweenness_centrality(Gs, weight="length", normalized=True)
def eb_de(u, v):
    return eb.get((u, v), eb.get((v, u), np.nan))

aristas = aristas.reset_index()
aristas["betweenness_arista"] = [eb_de(u, v) for u, v in zip(aristas["u"], aristas["v"])]

print("Top 5 nodos por intermediación:")
display(nodos.sort_values("betweenness", ascending=False)[["grado", "betweenness", "closeness"]].head())

## Fase 4 · Incorporación de variables de exposición
La exposición mide **cuánta gente e infraestructura sensible** hay alrededor de cada nodo. En el piloto se usan dos proxies abiertos de OpenStreetMap:

- **Puntos de interés (POI) ponderados**: hospitales, clínicas, escuelas, universidades, policía, bomberos, mercados, terminales (suma de pesos dentro del radio).
- **Densidad edificada**: cantidad de edificaciones dentro del radio (aproxima población y actividad).

Ambas se calculan **por nodo** y luego se asigna a cada arista el promedio de sus extremos.

In [ ]:
def descargar_features(tags):
    try:
        gdf = ox.features.features_from_point(CENTRO, tags=tags, dist=RADIO_M + RADIO_EXPOSICION_M)
        return gdf
    except Exception as e:  # sin resultados o error de red
        print("Aviso:", e)
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")

tags_poi = {"amenity": ["hospital", "clinic", "school", "college", "university",
                        "police", "fire_station", "marketplace", "bus_station"]}
poi = descargar_features(tags_poi)
edif = descargar_features({"building": True})
print(f"POIs encontrados: {len(poi)} | Edificaciones: {len(edif)}")

In [ ]:
def a_puntos_proyectados(gdf):
    if len(gdf) == 0:
        return gdf
    g = gdf.to_crs(CRS_M).copy()
    g["geometry"] = g.geometry.centroid
    return g

poi_p = a_puntos_proyectados(poi)
edif_p = a_puntos_proyectados(edif)

if len(poi_p):
    poi_p["peso"] = poi_p["amenity"].map(PESO_POI).fillna(1.0)

buf = nodos[["geometry"]].copy()
buf["geometry"] = buf.geometry.buffer(RADIO_EXPOSICION_M)
buf = buf.reset_index(names="node_id")

def suma_en_buffer(puntos, col=None):
    if len(puntos) == 0:
        return pd.Series(0.0, index=nodos.index)
    pts = puntos[["geometry"] + ([col] if col else [])].copy()
    if not col:
        pts["_u"] = 1.0
        col = "_u"
    j = gpd.sjoin(pts, buf, predicate="within", how="inner")
    return j.groupby("node_id")[col].sum().reindex(nodos.index).fillna(0.0)

nodos["expo_poi"] = suma_en_buffer(poi_p, "peso" if len(poi_p) else None)
nodos["expo_edificios"] = suma_en_buffer(edif_p)

nodos[["expo_poi", "expo_edificios"]].describe().round(2)

#### (Opcional) Incorporar tus propios datos de exposición
Si tienes un CSV con **densidad poblacional por sector (INEC)** o por punto (`lat, lon, valor`), puedes subirlo a Colab y activarlo aquí. Se suma como un tercer componente de exposición.

In [ ]:
# Cambia a True y ajusta el nombre del archivo si subes tu propio CSV (columnas: lat, lon, valor)
USAR_CSV_EXPOSICION = False
RUTA_CSV_EXPOSICION = "exposicion_inec.csv"

if USAR_CSV_EXPOSICION:
    df = pd.read_csv(RUTA_CSV_EXPOSICION)
    pts = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs="EPSG:4326").to_crs(CRS_M)
    nodos["expo_extra"] = suma_en_buffer(pts, "valor")
    print("Exposición adicional incorporada.")

In [ ]:
# Exposición por arista = promedio de los dos extremos
aristas["expo_poi"] = [(nodos.at[u, "expo_poi"] + nodos.at[v, "expo_poi"]) / 2 for u, v in zip(aristas["u"], aristas["v"])]
aristas["expo_edificios"] = [(nodos.at[u, "expo_edificios"] + nodos.at[v, "expo_edificios"]) / 2 for u, v in zip(aristas["u"], aristas["v"])]
print("Exposición asignada a nodos y aristas.")

## Fase 5 · Índice de criticidad y análisis estadístico

### 5.1 Índice de criticidad
Todas las variables se llevan a escala 0–1 (min–max) para que sean comparables:

$$\text{Estructural}=w_b\,B+w_c\,C+w_g\,G \qquad \text{Exposición}=w_p\,P+w_e\,E$$
$$\text{Criticidad}=\alpha\,\text{Estructural}+\beta\,\text{Exposición}$$

Un nodo es crítico cuando es **importante para la red** *y* **está donde hay gente e infraestructura sensible**. Los niveles Bajo/Medio/Alto se asignan por terciles.

In [ ]:
def minmax(s):
    rango = s.max() - s.min()
    return (s - s.min()) / rango if rango > 0 else s * 0

n = nodos
n["bet_n"], n["clo_n"], n["deg_n"] = minmax(n["betweenness"]), minmax(n["closeness"]), minmax(n["grado"])
n["poi_n"], n["edi_n"] = minmax(n["expo_poi"]), minmax(n["expo_edificios"])

n["indice_estructural"] = W_BET * n["bet_n"] + W_CLO * n["clo_n"] + W_DEG * n["deg_n"]
n["indice_exposicion"] = W_POI * n["poi_n"] + W_EDI * n["edi_n"]
n["criticidad"] = PESO_ESTRUCTURAL * n["indice_estructural"] + PESO_EXPOSICION * n["indice_exposicion"]

n["nivel_riesgo"] = pd.qcut(n["criticidad"].rank(method="first"), 3, labels=["Bajo", "Medio", "Alto"])
nodos = n

print(nodos["nivel_riesgo"].value_counts().reindex(["Alto", "Medio", "Bajo"]))
print("\nRanking: 10 nodos más críticos")
display(nodos.sort_values("criticidad", ascending=False)[["x", "y", "grado", "betweenness", "expo_poi", "expo_edificios", "criticidad", "nivel_riesgo"]].head(10).round(4))

In [ ]:
# Criticidad por arista (promedio de extremos del índice y betweenness propio normalizado)
aristas["crit_nodos"] = [(nodos.at[u, "criticidad"] + nodos.at[v, "criticidad"]) / 2 for u, v in zip(aristas["u"], aristas["v"])]
aristas["bet_arista_n"] = minmax(aristas["betweenness_arista"].fillna(0))
aristas["expo_arista_n"] = (minmax(aristas["expo_poi"]) * W_POI + minmax(aristas["expo_edificios"]) * W_EDI)
aristas["criticidad"] = PESO_ESTRUCTURAL * aristas["bet_arista_n"] + PESO_EXPOSICION * aristas["expo_arista_n"]
aristas["nivel_riesgo"] = pd.qcut(aristas["criticidad"].rank(method="first"), 3, labels=["Bajo", "Medio", "Alto"])
display(aristas.sort_values("criticidad", ascending=False)[["name", "highway", "length", "betweenness_arista", "criticidad", "nivel_riesgo"]].head(10).round(4))

### 5.2 Correlación de Spearman
Mide la **relación monótona** entre variables sin suponer normalidad. Responde a una pregunta clave: *¿los nodos estructuralmente importantes coinciden con los de mayor exposición?* Una correlación baja indicaría que la exposición aporta información que la topología sola no captura.

In [ ]:
vars_sp = ["grado", "betweenness", "closeness", "expo_poi", "expo_edificios", "criticidad"]
rho = nodos[vars_sp].corr(method="spearman")

fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(rho, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(vars_sp))); ax.set_xticklabels(vars_sp, rotation=45, ha="right")
ax.set_yticks(range(len(vars_sp))); ax.set_yticklabels(vars_sp)
for i in range(len(vars_sp)):
    for j in range(len(vars_sp)):
        ax.text(j, i, f"{rho.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, label="ρ de Spearman"); ax.set_title("Fase 5 · Correlación de Spearman entre variables")
plt.tight_layout(); plt.show()

r, p = spearmanr(nodos["betweenness"], nodos["expo_edificios"])
print(f"Betweenness vs edificaciones: ρ = {r:.3f} (p = {p:.3g})")

### 5.3 Regresión logística (Alto / Medio / Bajo)
Se entrena un modelo multiclase que clasifica el nivel de riesgo a partir de las variables del grafo y de exposición, con **validación cruzada estratificada**.

⚠️ **Lectura correcta:** como las etiquetas provienen del índice, un buen desempeño era esperable. Este bloque deja listo el *pipeline* (escalado → modelo → métricas) para sustituir la etiqueta por **eventos reales** en la siguiente celda.

In [ ]:
FEATURES = ["grado", "betweenness", "closeness", "expo_poi", "expo_edificios"]
X = nodos[FEATURES].values
y = nodos["nivel_riesgo"].astype(str).values

modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEMILLA))

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEMILLA)
modelo.fit(X_tr, y_tr)
pred = modelo.predict(X_te)

print(classification_report(y_te, pred, labels=["Bajo", "Medio", "Alto"]))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
acc = cross_val_score(modelo, X, y, cv=cv, scoring="accuracy")
print(f"Exactitud (validación cruzada 5 pliegues): {acc.mean():.3f} ± {acc.std():.3f}")

ConfusionMatrixDisplay(confusion_matrix(y_te, pred, labels=["Bajo", "Medio", "Alto"]),
                       display_labels=["Bajo", "Medio", "Alto"]).plot(cmap="Blues")
plt.title("Matriz de confusión (conjunto de prueba)"); plt.show()

coef = pd.DataFrame(modelo[-1].coef_, index=modelo[-1].classes_, columns=FEATURES).round(3)
print("Coeficientes (variables estandarizadas):"); display(coef)

#### (Opcional) Validación con eventos reales
Con un CSV de eventos (accidentes ATM, delitos ECU911; columnas `lat, lon`) se cuenta cuántos eventos caen cerca de cada nodo y se compara con la criticidad. Esta es la **validación real** del índice. Recuerda: pocos registros ≠ poco riesgo (subregistro).

In [ ]:
USAR_EVENTOS = False
FUENTE_EVENTOS = "supabase"           # "supabase" (df_accidentes) o "csv"
RUTA_EVENTOS = "accidentes_atm.csv"   # si FUENTE_EVENTOS = "csv"; usa las columnas COL_LAT / COL_LON

if USAR_EVENTOS:
    ev = df_accidentes.copy() if FUENTE_EVENTOS == "supabase" else pd.read_csv(RUTA_EVENTOS)
    ev[COL_LAT] = pd.to_numeric(ev[COL_LAT], errors="coerce")
    ev[COL_LON] = pd.to_numeric(ev[COL_LON], errors="coerce")
    ev = ev.dropna(subset=[COL_LAT, COL_LON])
    ev = gpd.GeoDataFrame(ev, geometry=gpd.points_from_xy(ev[COL_LON], ev[COL_LAT]), crs="EPSG:4326").to_crs(CRS_M)
    # Conservar solo los eventos dentro del área de estudio
    ev = ev[ev.within(nodos.unary_union.convex_hull.buffer(RADIO_EXPOSICION_M))]
    print(f"Eventos dentro de la zona: {len(ev):,}")
    nodos["eventos"] = suma_en_buffer(ev)
    r, p = spearmanr(nodos["criticidad"], nodos["eventos"])
    print(f"Criticidad vs eventos observados: ρ = {r:.3f} (p = {p:.3g})")
    # Etiqueta basada en eventos: percentiles de la tasa observada
    nodos["nivel_eventos"] = pd.qcut(nodos["eventos"].rank(method="first"), 3, labels=["Bajo", "Medio", "Alto"])

## Fase 6 · Visualización y mapas temáticos
Mapas para comunicar los resultados: criticidad, intermediación y exposición.

In [ ]:
def mapa_nodos(col, titulo, cmap="YlOrRd", nombre=None):
    fig, ax = plt.subplots(figsize=(8, 8))
    aristas.plot(ax=ax, color="#cccccc", linewidth=0.6)
    nodos.sort_values(col).plot(ax=ax, column=col, cmap=cmap, markersize=14, legend=True,
                                legend_kwds={"shrink": 0.6, "label": col})
    ax.set_title(titulo); ax.set_axis_off(); plt.tight_layout()
    if nombre:
        plt.savefig(f"{SALIDA}/{nombre}.png", dpi=200, bbox_inches="tight")
    plt.show()

mapa_nodos("criticidad", f"Índice de criticidad — {NOMBRE_ZONA}", nombre="mapa_criticidad")
mapa_nodos("betweenness", "Centralidad de intermediación", "viridis", "mapa_betweenness")
mapa_nodos("indice_exposicion", "Índice de exposición", "PuRd", "mapa_exposicion")

In [ ]:
# Mapa de tramos críticos y nodos de nivel Alto
colores = {"Bajo": "#9ecae1", "Medio": "#fdae6b", "Alto": "#d7301f"}
fig, ax = plt.subplots(figsize=(8, 8))
for nivel, c in colores.items():
    sub = aristas[aristas["nivel_riesgo"] == nivel]
    sub.plot(ax=ax, color=c, linewidth={"Bajo": 0.8, "Medio": 1.6, "Alto": 3.0}[nivel], label=f"Tramo {nivel}")
nodos[nodos["nivel_riesgo"] == "Alto"].plot(ax=ax, color="black", markersize=10, label="Nodo Alto")
ax.legend(loc="lower left"); ax.set_axis_off(); ax.set_title("Tramos y nodos por nivel de riesgo")
plt.tight_layout(); plt.savefig(f"{SALIDA}/mapa_niveles.png", dpi=200, bbox_inches="tight"); plt.show()

In [ ]:
# Mapa interactivo (se puede hacer zoom y ver el valor de cada nodo)
nodos_ll = nodos.to_crs(4326).reset_index().rename(columns={"osmid": "id_nodo"})
nodos_ll["geometry"] = nodos_ll.geometry
nodos_ll[["id_nodo", "criticidad", "nivel_riesgo", "grado", "betweenness", "expo_poi", "expo_edificios", "geometry"]].explore(
    column="criticidad", cmap="YlOrRd", tiles="CartoDB positron", marker_kwds={"radius": 5}, legend=True)

## Fase 7 · Evaluación y validación del artefacto
Siguiendo DSR, el artefacto se **evalúa**, no solo se construye. Dos pruebas:

**7.1 Prueba de robustez (simulación de falla).** Se retiran los nodos más críticos y se mide cuánto cae la **eficiencia** de la red (promedio de 1/distancia entre pares) y la fracción del **componente conexo gigante**. Si el índice es útil, quitar los nodos críticos debe degradar la red **más** que quitar el mismo número de nodos al azar.

In [ ]:
rng = np.random.default_rng(SEMILLA)
fuentes = rng.choice(list(Gs.nodes), size=min(250, Gs.number_of_nodes()), replace=False)

def eficiencia(H, fuentes):
    """Eficiencia media (1/d) desde un conjunto fijo de fuentes hacia todos los nodos de la red original."""
    total, pares = 0.0, 0
    N = Gs.number_of_nodes()
    for s in fuentes:
        if s not in H:
            continue
        d = nx.single_source_dijkstra_path_length(H, s, weight="length")
        total += sum(1.0 / v for k, v in d.items() if k != s and v > 0)
        pares += N - 1
    return total / pares

def evaluar(H):
    gig = max(nx.connected_components(H), key=len) if H.number_of_nodes() else []
    return eficiencia(H, fuentes), len(gig) / Gs.number_of_nodes()

base_ef, base_gig = evaluar(Gs)
fracciones = [0.02, 0.05, 0.10, 0.15]
filas = []
orden_crit = nodos.sort_values("criticidad", ascending=False).index
orden_bet = nodos.sort_values("betweenness", ascending=False).index

for f in fracciones:
    k = max(1, int(f * Gs.number_of_nodes()))
    for nombre, quitar in [("Criticidad (índice)", orden_crit[:k]), ("Solo betweenness", orden_bet[:k])]:
        H = Gs.copy(); H.remove_nodes_from(quitar)
        ef, gig = evaluar(H)
        filas.append((f, nombre, 1 - ef / base_ef, 1 - gig))
    azar_ef, azar_gig = [], []
    for _ in range(10):
        H = Gs.copy(); H.remove_nodes_from(rng.choice(list(Gs.nodes), size=k, replace=False))
        ef, gig = evaluar(H); azar_ef.append(1 - ef / base_ef); azar_gig.append(1 - gig)
    filas.append((f, "Azar (media de 10)", np.mean(azar_ef), np.mean(azar_gig)))

rob = pd.DataFrame(filas, columns=["fraccion_removida", "estrategia", "perdida_eficiencia", "perdida_componente_gigante"])
tabla = rob.pivot(index="fraccion_removida", columns="estrategia", values="perdida_eficiencia")
display(tabla.round(3))

tabla.plot(marker="o", figsize=(7, 4)); plt.ylabel("Pérdida de eficiencia"); plt.xlabel("Fracción de nodos removidos")
plt.title("Fase 7 · Robustez: remover nodos críticos vs. azar"); plt.grid(alpha=0.3); plt.show()

**7.2 Verificaciones de coherencia y reproducibilidad.**

In [ ]:
checks = {
    "Grafo conexo (no dirigido)": nx.is_connected(Gs),
    "Sin nodos aislados": nx.number_of_isolates(Gs) == 0,
    "Todas las aristas tienen longitud > 0": bool((aristas["length"] > 0).all()),
    "Métricas sin valores nulos": bool(nodos[["grado", "betweenness", "closeness", "criticidad"]].notna().all().all()),
    "Criticidad dentro de [0, 1]": bool(nodos["criticidad"].between(0, 1).all()),
    "Semilla fija (reproducible)": SEMILLA == 42,
}
pd.Series(checks).map({True: "✔ cumple", False: "✘ revisar"}).to_frame("estado")

## Exportación de entregables
Se guardan los productos que consumirán los bloques posteriores de RISKTWIN:

| Archivo | Uso |
|---|---|
| `grafo_urb02.graphml` | Grafo para URB-03/04/05 |
| `urb02_capas.gpkg` | Nodos y tramos con métricas (SIG, QGIS) |
| `nodos_metricas.csv` | Tabla para el índice de criticidad y la IA |
| `aristas_metricas.csv` | Tabla de tramos |
| `mapa_*.png` | Figuras para el informe |

In [ ]:
ox.io.save_graphml(Gp, f"{SALIDA}/grafo_urb02.graphml")

nodos_out = nodos.reset_index().rename(columns={"osmid": "id_nodo"})
aristas_out = aristas.copy()
cols_g = ["u", "v", "key", "name", "highway", "length", "betweenness_arista", "expo_poi", "expo_edificios", "criticidad", "nivel_riesgo", "geometry"]
nodos_out.to_crs(4326).to_file(f"{SALIDA}/urb02_capas.gpkg", layer="nodos", driver="GPKG")
aristas_out[cols_g].astype({"name": str, "highway": str}).to_crs(4326).to_file(f"{SALIDA}/urb02_capas.gpkg", layer="aristas", driver="GPKG")
nodos_out.drop(columns="geometry").to_csv(f"{SALIDA}/nodos_metricas.csv", index=False)
aristas_out[cols_g].drop(columns="geometry").to_csv(f"{SALIDA}/aristas_metricas.csv", index=False)
rob.to_csv(f"{SALIDA}/robustez.csv", index=False)

print(sorted(os.listdir(SALIDA)))

In [ ]:
# Descarga todo en un .zip desde Colab (ignorar si no estás en Colab)
try:
    import shutil
    from google.colab import files
    shutil.make_archive("salidas_urb02", "zip", SALIDA)
    files.download("salidas_urb02.zip")
except ImportError:
    print("Fuera de Colab: los archivos están en la carpeta", SALIDA)

### (Opcional) Guardar resultados en Supabase
Sube las tablas de nodos y tramos a PostgreSQL para que las consuman URB-03/04/05 y la API. **Las tablas deben existir antes** (puedes crearlas desde el *SQL Editor* de Supabase, o dejar que se ajusten a estas columnas):

```sql
create table urb02_nodos (
  id_nodo bigint, x double precision, y double precision, grado int,
  betweenness double precision, closeness double precision,
  expo_poi double precision, expo_edificios double precision,
  indice_estructural double precision, indice_exposicion double precision,
  criticidad double precision, nivel_riesgo text
);
create table urb02_aristas (
  u bigint, v bigint, key int, name text, highway text, length double precision,
  betweenness_arista double precision, expo_poi double precision,
  expo_edificios double precision, criticidad double precision, nivel_riesgo text
);
```
Solo se envían columnas de la tabla; ajusta la lista si tu esquema difiere. Ejecutar esta celda dos veces duplica filas.

In [ ]:
if USAR_SUPABASE:
    cols_nodos = ["id_nodo", "x", "y", "grado", "betweenness", "closeness", "expo_poi", "expo_edificios",
                  "indice_estructural", "indice_exposicion", "criticidad", "nivel_riesgo"]
    cols_aristas = ["u", "v", "key", "name", "highway", "length", "betweenness_arista",
                    "expo_poi", "expo_edificios", "criticidad", "nivel_riesgo"]
    guardar_en_supabase(nodos_out[cols_nodos], TABLA_SALIDA_NODOS)
    guardar_en_supabase(aristas_out[cols_aristas], TABLA_SALIDA_ARISTAS)
else:
    print("Supabase desactivado: los resultados quedaron solo en la carpeta", SALIDA)

## Conclusiones del piloto y siguientes pasos

**Qué demuestra este piloto**
- El flujo completo (OSM → grafo → métricas → exposición → criticidad → mapas → evaluación) es **reproducible, de bajo costo y automatizable**.
- La red y los índices quedan en formatos (GraphML, GeoPackage, CSV) listos para **URB-03, URB-04, URB-05 y SIS-API-01**.
- La prueba de robustez permite argumentar con evidencia si el índice identifica elementos realmente críticos.

**Limitaciones a declarar en la tesis**
- La exposición usa proxies de OSM; la completitud de OSM varía entre sectores.
- Las etiquetas Alto/Medio/Bajo del piloto son internas al índice (circulares); la validación definitiva requiere eventos reales.
- Los pesos del índice son un supuesto de diseño; conviene un **análisis de sensibilidad** (variar `PESO_ESTRUCTURAL`, `W_BET`, etc.).

**Siguientes pasos**
1. Fijar la zona definitiva y ampliar el radio o usar el polígono de un barrio.
2. Reemplazar los proxies por densidad poblacional del INEC (celda opcional de la Fase 4).
3. Cargar accidentes ATM / eventos ECU911 georreferenciados para validar el índice (celda opcional de la Fase 5).
4. Comparar la regresión logística con Random Forest o XGBoost y comparar los pesos del índice.
5. Entregar los archivos exportados a URB-03.